# Try applying Reinforcement Learning to Task scheduler

## 1. Exlore a graph

In [2]:
from pathlib import Path
import json
graph_path = Path("../benchmarks/mlsys-2026-1.json")

problem = json.loads(graph_path.read_text())
problem

{'widths': [512, 512, 512, 512, 512, 512, 512, 512, 512],
 'heights': [512, 512, 512, 512, 512, 512, 512, 512, 512],
 'inputs': [[0, 1], [4], [5, 2], [6, 3], [7, 0]],
 'outputs': [[4], [5], [6], [7], [8]],
 'base_costs': [2000, 500, 2000, 2000, 500],
 'op_types': ['MatMul', 'Pointwise', 'MatMul', 'MatMul', 'Pointwise'],
 'fast_memory_capacity': 60000,
 'slow_memory_bandwidth': 20,
 'native_granularity': [128, 128]}

In [6]:
for op_id, op_type in enumerate(problem["op_types"]):
    inputs = problem["inputs"][op_id]
    outputs = problem["outputs"][op_id]
    cost = problem["base_costs"][op_id]

    print(
        f"Operation {op_id}: {op_type} | "
        f"inputs={inputs} → outputs={outputs} | base_cost={cost}"
    )

Operation 0: MatMul | inputs=[0, 1] → outputs=[4] | base_cost=2000
Operation 1: Pointwise | inputs=[4] → outputs=[5] | base_cost=500
Operation 2: MatMul | inputs=[5, 2] → outputs=[6] | base_cost=2000
Operation 3: MatMul | inputs=[6, 3] → outputs=[7] | base_cost=2000
Operation 4: Pointwise | inputs=[7, 0] → outputs=[8] | base_cost=500


## 2. Try the evaluator

In [8]:
import sys

ROOT = graph_path.resolve().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
    
from evaluator import Evaluator

evaluator = Evaluator(problem)
evaluator


In [11]:
baseline = evaluator.baseline()
schedule = baseline["schedule"]
schedule

{'subgraphs': [[0], [1], [2], [3], [4]],
 'granularities': [[128, 128, 128],
  [128, 128, 1],
  [128, 128, 128],
  [128, 128, 128],
  [128, 128, 1]],
 'tensors_to_retain': [[], [], [], [], []],
 'subgraph_latencies': [117964.80000000002,
  26214.4,
  117964.80000000002,
  117964.80000000002,
  39321.600000000006]}

In [12]:
result = evaluator.evaluate(schedule)

print("Valid:", result["valid"])
print("Simulated latency:", result["total_latency"])
print("Memory per group:", result["working_sets"])

Valid: True
Simulated latency: 419430.4
Memory per group: [49152, 32768, 49152, 49152, 49152]


In [13]:
feedback = evaluator.reward(schedule)

print("Reward:", feedback["reward"])
print("Baseline latency:", feedback["baseline_latency"])

Reward: 0.5
Baseline latency: 419430.4


## 3. Define the RL environment

### Environment definition

**Goal:** learn how to group operations into a valid execution schedule that reduces simulated latency for a given graph and hardware configuration.

One episode plans a schedule for one fixed graph and hardware profile. The environment edits the plan and asks the evaluator to score it; it does not execute tensor computations on real hardware.

#### 1. Initial state and reset

- Start from `Evaluator.baseline()`: each operation is in a separate group, in a valid topological order, with feasible tiles.
- Keep an unchanged copy of that baseline and its latency, **B**.
- Create an independent working schedule using `deepcopy`.
- Set `boundary_index = 0` and clear the episode's completion flag.
- Remove stored `subgraph_latencies` from the working schedule before editing it; the evaluator will calculate fresh values.

For the current five-operation graph, the initial groups are `[[0], [1], [2], [3], [4]]`.

#### 2. Observation: what the agent receives

The observation represents information needed to choose a scheduling decision:

- **Graph:** operation types and base costs, ordered tensor dependencies, and tensor dimensions.
- **Hardware:** fast-memory capacity, slow-memory bandwidth, and native granularity.
- **Current plan:** operation-group membership and tile sizes.
- **Decision position:** the original boundary currently being considered and the decisions already made.

The environment stores the complete state. For a neural policy, encode the observation as numerical arrays with a declared Gymnasium `observation_space`; the raw nested schedule dictionary is useful for debugging but is not directly a numerical network input. When using multiple graph sizes, the encoding must account for those sizes, for example through padding and masks.

#### 3. Action: what the agent chooses

Use `gymnasium.spaces.Discrete(2)`:

- **0 — KEEP:** preserve the current boundary, so its groups remain separate.
- **1 — FUSE:** remove the current boundary and combine its adjacent groups.

Example: fusing the first boundary changes `[[0], [1], [2]]` into `[[0, 1], [2]]`.

For this first version, operation order is fixed and tensor retention is disabled. Tile selection follows a deterministic feasibility rule rather than being learned.

#### 4. Valid actions and tile selection

- KEEP is valid because the current complete schedule is already valid.
- For FUSE, construct a complete candidate schedule, including the unprocessed suffix.
- Try a predefined, ordered set of candidate tiles. Select the first candidate that passes the evaluator's tiling, dependency, tensor-availability, and memory checks.
- Expose `action_masks()` as a Boolean array: `[True, can_fuse]`.
- A valid fusion remains available even if it increases latency. Learning whether it is worthwhile belongs to the agent.

The tile catalog limits the search space: failure to find a tile in that catalog does not prove that every possible tile is invalid. Existing singleton tiles are not automatically valid after fusion.

#### 5. Transition: how an action changes the state

1. Copy the current schedule before proposing a fusion.
2. For FUSE, combine the groups and update their tile and retention entries. Accept the complete candidate only after validation.
3. For KEEP, preserve the current plan.
4. Advance to the next **original** boundary after either accepted action.
5. Return the new observation and step results. Cache the current state's candidate checks to avoid repeating them unnecessarily.

Every intermediate working plan stays complete and valid. This simplifies evaluator integration, but can exclude larger fusion groups that would require an invalid intermediate merge; that is a limitation of this initial action space.

#### 6. Termination: when an episode ends

- With **N** operations, finish after deciding the **N − 1** original boundaries.
- At completion, return `terminated=True` and export the final complete schedule.
- `truncated=True` is reserved for an external time or step limit, rather than normal completion.
- An invalid requested fusion, if one bypasses the mask, ends the episode with reward `-1`.
- Evaluator setup/backend failures raise errors; they are not agent mistakes and must not become invalid-action penalties.

#### 7. Reward: how success is measured

- **Intermediate steps:** reward is `0`.
- **Final valid schedule:** use the existing `Evaluator.reward(schedule)` result.

For baseline latency **B** and final schedule latency **L**:

`final_reward = B / (B + L)`

A baseline-equivalent schedule receives `0.5`; a faster one receives more than `0.5`; a slower valid one receives less than `0.5`. Invalid actions receive `-1`. All latencies are simulated values under the pinned evaluator model.

Use `gamma=1.0` as the initial undiscounted objective for these finite episodes. Start with terminal rewards so their meaning is easy to verify.

#### 8. Gymnasium interface and debugging information

- `reset(seed=None, options=None)` returns `(observation, info)` and restores the initial plan.
- `step(action)` returns `(observation, reward, terminated, truncated, info)`.
- `action_masks()` returns the current action-validity mask for MaskablePPO.
- `close()` releases the evaluator's native resources.
- `info` records diagnostics such as baseline/current latency, validity, group count, and evaluator-call count. These fields do not automatically become policy inputs.


As the first version of the environment, we just include the fusion action.

In [ ]:
from copy import deepcopy
import gymnasium as gym
from gymnasium import spaces
from evaluator import Evaluator
import numpy as np

class SchedulingEnv(gym.Env):
    def __init__(self, problem):
        super().__init__()
        
        self.problem = deepcopy(problem)
        self.evaluator = Evaluator(self.problem)
        
        baseline = self.evaluator.baseline()
        self.baseline_schedule = baseline["schedule"]
        # We dont need the subgraph latency because the agent will change it
        # the subgraph latency here belongs to the original graph
        self.baseline_schedule.pop("subgraph_latencies", None)
        self.baseline_latency = baseline["evaluation"]["total_latency"]

        # Observation space
        self.num_ops = len(self.problem["op_types"])
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_ops+1,),
            dtype=np.float32
        )
        
        # Action space
        self.action_space = spaces.Discrete(2)
        
        
    
    def get_observation(self):
        observation = np.zeros(self.num_ops + 1, dtype=np.float32)
        # Scale making group ID in [0,1]
        scale = max(1, self.num_ops - 1)

        for group_index, group in enumerate(self.schedule["subgraphs"]):
            for op in group:
                observation[op] = group_index / scale

        observation[-1] = self.boundary_index / scale

        return observation
        
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.schedule = deepcopy(self.baseline_schedule)
        self.boundary_index = 0
        # If there is only one operation then we dont need to fuse
        self.terminated = len(self.problem["op_types"]) <= 1
        
        observation = self.get_observation()
        
        info = {"baseline_latency": self.baseline_latency}
        
        return observation, info
    
    def create_fusion_candidate(self):
        '''
        [1] [2] | [3] [4] [5]
        
        i = 1
        [1] [2,3] [4] [5]
        '''
        candidate = deepcopy(self.schedule)
        i = self.boundary_index
        
        # Combine the adjacent groups
        left = candidate["subgraphs"][i]
        right = candidate["subgraphs"][i+1]
        
        merged = left + right
        candidate["subgraphs"][i:i+2] = [merged]
        
        # A simple trial tile for our current graph.
        # Check if the merged op having Matmul
        has_matmul = any(
            self.problem["op_types"][op] == "MatMul"
            for op in merged
        )
        tile = [128, 128, 64 if has_matmul else 1]

        candidate["granularities"][i:i + 2] = [tile]
        candidate["tensors_to_retain"][i:i + 2] = [[]]
        
        return candidate

    
    def step(self, action):
        if self.terminated:
            raise RuntimeError("Episode finished. Call reset() first")
        
        if not self.action_space.contains(action):
            raise ValueError("Action must be 0 or 1.")
        
        info = {"baseline_latency": self.baseline_latency}
        
        # Keep the boundary
        if action == 0:
            self.boundary_index += 1
        elif action == 1:
            candidate = self.create_fusion_candidate()
            evaluation = self.evaluator.evaluate(candidate)
            
            # If the candidate is not valid
            if not evaluation["valid"]:
                self.terminated = True
                info.update(evaluation)
                return self.get_observation(), -1.0, True, False, info

            # The candidate is valid
            self.schedule = candidate
            
            
        self.terminated = self.boundary_index >= len(self.schedule["subgraphs"]) - 1
        reward = 0.0

        if self.terminated:
            feedback = self.evaluator.reward(self.schedule)
            reward = feedback["reward"]
            info.update(feedback["evaluation"])
        
        return (
            self.get_observation(),
            reward,
            self.terminated,
            False,  # No external time limit, so not truncated.
            info
        )
    
    def close(self):
        self.evaluator.close()

In [43]:
env = SchedulingEnv(problem)

observation, info = env.reset()

while not env.terminated:
    # Choose keep
    observation, reward, terminated, truncated, info = env.step(0)

    print(
        "Boundary index:", observation["boundary_index"],
        "| Reward:", reward,
        "| Finished:", terminated,
    )

Boundary index: 1 | Reward: 0.0 | Finished: False
Boundary index: 2 | Reward: 0.0 | Finished: False
Boundary index: 3 | Reward: 0.0 | Finished: False
Boundary index: 4 | Reward: 0.5 | Finished: True


Test fuse step

In [55]:
env = SchedulingEnv(problem)

obs, info = env.reset()

# Fuse
obs, reward, terminated, truncated, info = env.step(1)
print("Groups:", obs["schedule"]["subgraphs"])
print("Boundary:", obs["boundary_index"])
print("Reward:", reward)
print("Finished:", terminated)

Groups: [[0, 1], [2], [3], [4]]
Boundary: 0
Reward: 0.0
Finished: False


Let's test a complete episode using two experiment. 

Keep everything seperate vs fuse the first two operations

In [60]:
env = SchedulingEnv(problem)

experiments = [
    ("Keep everything separate", [0, 0, 0, 0]),
    ("Fuse the first two operations", [1, 0, 0, 0]),
]

for name, actions in experiments:
    obs, info = env.reset()
    print(f"\n{name}")

    for action in actions:
        obs, reward, terminated, truncated, info = env.step(action)

        print(
            f"action={action}, "
            f"boundary={obs['boundary_index']}, "
            f"reward={reward:.4f}, "
            f"finished={terminated}"
        )

        if terminated or truncated:
            break

    print("Final groups:", obs["schedule"]["subgraphs"])
    print("Final latency:", info.get("total_latency"))


Keep everything separate
action=0, boundary=1, reward=0.0000, finished=False
action=0, boundary=2, reward=0.0000, finished=False
action=0, boundary=3, reward=0.0000, finished=False
action=0, boundary=4, reward=0.5000, finished=True
Final groups: [[0], [1], [2], [3], [4]]
Final latency: 419430.4

Fuse the first two operations
action=1, boundary=0, reward=0.0000, finished=False
action=0, boundary=1, reward=0.0000, finished=False
action=0, boundary=2, reward=0.0000, finished=False
action=0, boundary=3, reward=0.5161, finished=True
Final groups: [[0, 1], [2], [3], [4]]
Final latency: 393216.0


Next we have to have a way to encode the observation into numerical.

For example, right now, the observation looks like:

```python
{
    "schedule": {
        "subgraphs": [[0, 1], [2], [3], [4]],
        # other settings...
    },
    "boundary_index": 0,
}
```
Neural network can't understand this.
We need to encode the state as numbers:
```
Current groups:     [[0, 1], [2], [3], [4]]
Group assignments: [0,  0,   1,   2,   3]
Current boundary:  0
```

Let's test the new observation space

In [63]:
env = SchedulingEnv(problem)

obs, info = env.reset()
print("Before:", obs)

obs, reward, terminated, truncated, info = env.step(1)
print("After:", obs)

print("Matches observation space:", env.observation_space.contains(obs))

Before: [0.   0.25 0.5  0.75 1.   0.  ]
After: [0.   0.   0.25 0.5  0.75 0.  ]
Matches observation space: True


Check if our current environment compatiable with the training library

In [64]:
from stable_baselines3.common.env_checker import check_env

env = SchedulingEnv(problem)

check_env(env, warn=True)

print("Environment check passed!")

Environment check passed!


Now let's test the first training algorithm:

REINFORCE

First we have to build the policy network

In [140]:
import torch
from torch import nn
from torch.distributions import Categorical

torch.manual_seed(42)

env = SchedulingEnv(problem)

num_inputs = env.observation_space.shape[0] # 6
num_actions = env.action_space.n # 2 keep or fuse

# 6 -> 32 -> 2
policy = nn.Sequential(
    nn.Linear(num_inputs, 32),
    nn.Tanh(),
    nn.Linear(32, num_actions)
)



In [75]:
obs, info = env.reset()
print(info)
obs

{'baseline_latency': 419430.4}


array([0.  , 0.25, 0.5 , 0.75, 1.  , 0.  ], dtype=float32)

In [117]:
obs, info = env.reset()

obs_tensor = torch.as_tensor(obs, dtype=torch.float32)
logits = policy(obs_tensor)
print(logits)
distribution = Categorical(logits=logits)

action = distribution.sample()
log_prob = distribution.log_prob(action)

print("Probabilities [KEEP, FUSE]:", distribution.probs.detach().numpy())
print("Chosen action:", action.item())
print("Log probability:", log_prob.item())

tensor([ 0.4744, -0.1163], grad_fn=<UnsafeViewBackward0>)
Probabilities [KEEP, FUSE]: [0.64352673 0.35647327]
Chosen action: 0
Log probability: -0.4407917261123657


Forward pass

In [135]:
obs, info = env.reset()

log_probs = []
actions = []

terminated = False
truncated = False

while not (terminated or truncated):
    obs_tensor = torch.as_tensor(obs, dtype=torch.float32)

    logits = policy(obs_tensor)
    distribution = Categorical(logits=logits)

    action = distribution.sample()

    # Keep this tensor connected to the policy for training.
    log_probs.append(distribution.log_prob(action))
    actions.append(action.item())

    obs, reward, terminated, truncated, info = env.step(action.item())

final_reward = reward

print("Actions:", actions)
print("Final groups:", env.schedule["subgraphs"])
print("Final reward:", final_reward)
print("Final latency:", info.get("total_latency"))

Actions: [0, 0, 0, 0]
Final groups: [[0], [1], [2], [3], [4]]
Final reward: 0.5
Final latency: 419430.4


Now we need optimizer

In [137]:
optimizer = torch.optim.Adam(policy.parameters(), lr = 0.001)
# Combine the log probabilities of all actions in the episode.
episode_log_prob = torch.stack(log_probs).sum()

# REINFORCE loss: negative reward × episode log probability.
loss = -final_reward * episode_log_prob

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Episode reward:", final_reward)
print("Training loss:", loss.item())

Episode reward: 0.5
Training loss: 0.8395570516586304


Let's start training

In [ ]:
num_episodes = 1000
reward_history = []

for episode in range(num_episodes):
    obs, info = env.reset()

    log_probs = []
    terminated = False
    truncated = False

    # 1. Collect one complete episode.
    while not (terminated or truncated):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)

        logits = policy(obs_tensor)
        distribution = Categorical(logits=logits)

        action = distribution.sample()
        log_probs.append(distribution.log_prob(action))

        obs, reward, terminated, truncated, info = env.step(
            action.item()
        )

    # Intermediate rewards are zero, so use the final reward.
    final_reward = reward

    # 2. Update the policy using this episode.
    loss = -final_reward * torch.stack(log_probs).sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 3. Record progress.
    reward_history.append(final_reward)

    if (episode + 1) % 100 == 0:
        mean_reward = sum(reward_history[-100:]) / 100

        print(
            f"Episode {episode + 1}: "
            f"average reward = {mean_reward:.4f}"
        )

Episode 100: average reward = 0.0058
Episode 200: average reward = 0.4403
Episode 300: average reward = 0.4703
Episode 400: average reward = 0.4702
Episode 500: average reward = 0.5000
Episode 600: average reward = 0.4700
Episode 700: average reward = 0.5000
Episode 800: average reward = 0.5000
Episode 900: average reward = 0.4700
Episode 1000: average reward = 0.5000


In [139]:
obs, info = env.reset()
actions = []
terminated = False
truncated = False

with torch.no_grad():
    while not (terminated or truncated):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)
        logits = policy(obs_tensor)

        # Choose the most probable action instead of sampling.
        action = logits.argmax().item()
        actions.append(action)

        obs, reward, terminated, truncated, info = env.step(action)

print("Actions:", actions)  # 0 = KEEP, 1 = FUSE
print("Final groups:", env.schedule["subgraphs"])
print("Final reward:", reward)
print("Final latency:", info.get("total_latency"))
print("Baseline latency:", env.baseline_latency)

Actions: [0, 0, 0, 0]
Final groups: [[0], [1], [2], [3], [4]]
Final reward: 0.5
Final latency: 419430.4
Baseline latency: 419430.4


The agent only choose keep. Which is safe, exploitation.

In [145]:
import torch
from torch import nn
from torch.distributions import Categorical

torch.manual_seed(42)

env = SchedulingEnv(problem)

num_inputs = env.observation_space.shape[0] # 6
num_actions = env.action_space.n # 2 keep or fuse

# 6 -> 32 -> 2
policy = nn.Sequential(
    nn.Linear(num_inputs, 32),
    nn.Tanh(),
    nn.Linear(32, num_actions)
)

optimizer = torch.optim.Adam(policy.parameters(), lr=0.001)
num_episodes = 1000
reward_history = []

running_baseline = 0.5  # Reward of our separate schedule.
baseline_decay = 0.95

for episode in range(num_episodes):
    obs, info = env.reset()

    log_probs = []
    terminated = False
    truncated = False

    # 1. Collect one complete episode.
    while not (terminated or truncated):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)

        logits = policy(obs_tensor)
        distribution = Categorical(logits=logits)

        action = distribution.sample()
        log_probs.append(distribution.log_prob(action))

        obs, reward, terminated, truncated, info = env.step(
            action.item()
        )

    final_reward = reward

    # Compare this episode with previous performance.
    advantage = final_reward - running_baseline

    loss = -advantage * torch.stack(log_probs).sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Update the baseline AFTER calculating this episode's advantage.
    running_baseline = (
        baseline_decay * running_baseline
        + (1 - baseline_decay) * final_reward
    )

    # 3. Record progress.
    reward_history.append(final_reward)

    if (episode + 1) % 100 == 0:
        mean_reward = sum(reward_history[-100:]) / 100

        print(
            f"Episode {episode + 1}: "
            f"average reward = {mean_reward:.4f}"
        )

Episode 100: average reward = -0.1290
Episode 200: average reward = 0.4405
Episode 300: average reward = 0.4553
Episode 400: average reward = 0.4702
Episode 500: average reward = 0.5000
Episode 600: average reward = 0.4850
Episode 700: average reward = 0.4850
Episode 800: average reward = 0.5000
Episode 900: average reward = 0.5000
Episode 1000: average reward = 0.4700


It fail to use fuse because everysingle time it try fuse it usually have -1 reward/

We can use an action mask to guide the agent here.

In [149]:
from copy import deepcopy
import gymnasium as gym
from gymnasium import spaces
from evaluator import Evaluator
import numpy as np

class SchedulingEnv(gym.Env):
    def __init__(self, problem):
        super().__init__()
        
        self.problem = deepcopy(problem)
        self.evaluator = Evaluator(self.problem)
        
        baseline = self.evaluator.baseline()
        self.baseline_schedule = baseline["schedule"]
        # We dont need the subgraph latency because the agent will change it
        # the subgraph latency here belongs to the original graph
        self.baseline_schedule.pop("subgraph_latencies", None)
        self.baseline_latency = baseline["evaluation"]["total_latency"]

        # Observation space
        self.num_ops = len(self.problem["op_types"])
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_ops+1,),
            dtype=np.float32
        )
        
        # Action space
        self.action_space = spaces.Discrete(2)
    
    def action_masks(self):
        if self.terminated:
            return np.array([False, False], dtype=bool)

        candidate = self.create_fusion_candidate()
        evaluation = self.evaluator.evaluate(candidate)

        return np.array(
            [True, evaluation["valid"]],  # [KEEP, FUSE]
            dtype=bool,
        )
    def get_observation(self):
        observation = np.zeros(self.num_ops + 1, dtype=np.float32)
        # Scale making group ID in [0,1]
        scale = max(1, self.num_ops - 1)

        for group_index, group in enumerate(self.schedule["subgraphs"]):
            for op in group:
                observation[op] = group_index / scale

        observation[-1] = self.boundary_index / scale

        return observation
        
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.schedule = deepcopy(self.baseline_schedule)
        self.boundary_index = 0
        # If there is only one operation then we dont need to fuse
        self.terminated = len(self.problem["op_types"]) <= 1
        
        observation = self.get_observation()
        
        info = {"baseline_latency": self.baseline_latency}
        
        return observation, info
    
    def create_fusion_candidate(self):
        '''
        [1] [2] | [3] [4] [5]
        
        i = 1
        [1] [2,3] [4] [5]
        '''
        candidate = deepcopy(self.schedule)
        i = self.boundary_index
        
        # Combine the adjacent groups
        left = candidate["subgraphs"][i]
        right = candidate["subgraphs"][i+1]
        
        merged = left + right
        candidate["subgraphs"][i:i+2] = [merged]
        
        # A simple trial tile for our current graph.
        # Check if the merged op having Matmul
        has_matmul = any(
            self.problem["op_types"][op] == "MatMul"
            for op in merged
        )
        tile = [128, 128, 64 if has_matmul else 1]

        candidate["granularities"][i:i + 2] = [tile]
        candidate["tensors_to_retain"][i:i + 2] = [[]]
        
        return candidate

    
    def step(self, action):
        if self.terminated:
            raise RuntimeError("Episode finished. Call reset() first")
        
        if not self.action_space.contains(action):
            raise ValueError("Action must be 0 or 1.")
        
        info = {"baseline_latency": self.baseline_latency}
        
        # Keep the boundary
        if action == 0:
            self.boundary_index += 1
        elif action == 1:
            candidate = self.create_fusion_candidate()
            evaluation = self.evaluator.evaluate(candidate)
            
            # If the candidate is not valid
            if not evaluation["valid"]:
                self.terminated = True
                info.update(evaluation)
                return self.get_observation(), -1.0, True, False, info

            # The candidate is valid
            self.schedule = candidate
            
            
        self.terminated = self.boundary_index >= len(self.schedule["subgraphs"]) - 1
        reward = 0.0

        if self.terminated:
            feedback = self.evaluator.reward(self.schedule)
            reward = feedback["reward"]
            info.update(feedback["evaluation"])
        
        return (
            self.get_observation(),
            reward,
            self.terminated,
            False,  # No external time limit, so not truncated.
            info
        )
    
    def close(self):
        self.evaluator.close()

In [227]:
import torch
from torch import nn
from torch.distributions import Categorical
from pathlib import Path
import json
import sys

ROOT = graph_path.resolve().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from evaluator import Evaluator

torch.manual_seed(42)

graph_path = Path("../benchmarks/mlsys-2026-5.json")
problem = json.loads(graph_path.read_text())

evaluator = Evaluator(problem)
env = SchedulingEnv(problem)

num_inputs = env.observation_space.shape[0] # 6
num_actions = env.action_space.n # 2 keep or fuse

# 6 -> 32 -> 2
policy = nn.Sequential(
    nn.Linear(num_inputs, 32),
    nn.Tanh(),
    nn.Linear(32, num_actions)
)

optimizer = torch.optim.Adam(policy.parameters(), lr=0.001)
num_episodes = 6200
reward_history = []

running_baseline = 0.5  # Reward of our separate schedule.
baseline_decay = 0.95

best_schedule = deepcopy(env.baseline_schedule)
best_reward = 0.5
best_latency = env.baseline_latency
print(best_latency)

for episode in range(num_episodes):
    obs, info = env.reset()

    log_probs = []
    terminated = False
    truncated = False

    # 1. Collect one complete episode.
    while not (terminated or truncated):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)

        logits = policy(obs_tensor)
        
        mask = torch.as_tensor(
            env.action_masks(),
            dtype=torch.bool,
            device=logits.device,
        )
        masked_logits = logits.masked_fill(~mask, float("-inf"))
        
        distribution = Categorical(logits=masked_logits)

        action = distribution.sample()
        log_probs.append(distribution.log_prob(action))

        obs, reward, terminated, truncated, info = env.step(
            action.item()
        )

    final_reward = reward
    # record the best schedule 
    if info.get("valid", False) and final_reward > best_reward:
        best_reward = final_reward
        best_schedule = deepcopy(env.schedule)
        best_latency = info["total_latency"]

    # Compare this episode with previous performance.
    advantage = final_reward - running_baseline

    loss = -advantage * torch.stack(log_probs).sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Update the baseline AFTER calculating this episode's advantage.
    running_baseline = (
        baseline_decay * running_baseline
        + (1 - baseline_decay) * final_reward
    )

    # 3. Record progress.
    reward_history.append(final_reward)

    if (episode + 1) % 100 == 0:
        mean_reward = sum(reward_history[-100:]) / 100

        print(
            f"Episode {episode + 1}: "
            f"average reward = {mean_reward:.4f}"
        )
print("Best groups:", best_schedule["subgraphs"])
print("Best reward:", best_reward)
print("Best latency:", best_latency)

1336516.266666666
Episode 100: average reward = 0.5000
Episode 200: average reward = 0.5000
Episode 300: average reward = 0.5000
Episode 400: average reward = 0.5000
Episode 500: average reward = 0.5000
Episode 600: average reward = 0.5000
Episode 700: average reward = 0.5000
Episode 800: average reward = 0.5000
Episode 900: average reward = 0.5000
Episode 1000: average reward = 0.5000
Episode 1100: average reward = 0.5000
Episode 1200: average reward = 0.5000
Episode 1300: average reward = 0.5000
Episode 1400: average reward = 0.5000
Episode 1500: average reward = 0.5000
Episode 1600: average reward = 0.5000
Episode 1700: average reward = 0.5000
Episode 1800: average reward = 0.5000
Episode 1900: average reward = 0.5000
Episode 2000: average reward = 0.5000
Episode 2100: average reward = 0.5000
Episode 2200: average reward = 0.5000
Episode 2300: average reward = 0.5000
Episode 2400: average reward = 0.5000
Episode 2500: average reward = 0.5000
Episode 2600: average reward = 0.5000
Epi

In [220]:
obs, info = env.reset()
actions = []
done = False

with torch.no_grad():
    while not done:
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)
        logits = policy(obs_tensor)

        mask = torch.as_tensor(
            env.action_masks(),
            dtype=torch.bool,
            device=logits.device,
        )

        masked_logits = logits.masked_fill(~mask, float("-inf"))
        distribution = Categorical(logits=masked_logits)

        print(
            "Allowed actions [KEEP, FUSE]:", mask.tolist(),
            "Probabilities:", distribution.probs.tolist(),
        )

        action = distribution.probs.argmax().item()
        actions.append(action)

        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

print("Chosen actions:", actions)
print("Reward:", reward)
print("Latency:", info.get("total_latency"))

Allowed actions [KEEP, FUSE]: [True, True] Probabilities: [6.24913809588179e-06, 0.9999938011169434]
Allowed actions [KEEP, FUSE]: [True, False] Probabilities: [1.0, 0.0]
Allowed actions [KEEP, FUSE]: [True, False] Probabilities: [1.0, 0.0]
Allowed actions [KEEP, FUSE]: [True, False] Probabilities: [1.0, 0.0]
Chosen actions: [1, 0, 0, 0]
Reward: 0.5161290322580645
Latency: 393216.0


Let's try PPO

In [221]:
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

torch.set_num_threads(1)

ppo_env = Monitor(SchedulingEnv(problem))

ppo_model = PPO(
    "MlpPolicy",
    ppo_env,
    learning_rate=0.0003,
    n_steps=128,
    batch_size=32,
    n_epochs=10,
    gamma=1.0,
    gae_lambda=1.0,
    ent_coef=0.01,
    policy_kwargs={
        "net_arch": {"pi": [32], "vf": [32]}
    },
    seed=42,
    device="cpu",
    verbose=1,
)

ppo_model.learn(total_timesteps=24_576)

Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.84     |
|    ep_rew_mean     | -0.766   |
| time/              |          |
|    fps             | 5372     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 128      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 2.85         |
|    ep_rew_mean          | -0.78        |
| time/                   |              |
|    fps                  | 4162         |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 256          |
| train/                  |              |
|    approx_kl            | 0.0048590847 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.691       |
|    e

In [222]:
eval_env = SchedulingEnv(problem)
obs, info = eval_env.reset()

actions = []
done = False

while not done:
    action, _ = ppo_model.predict(obs, deterministic=True)
    action = int(action.item())
    actions.append(action)

    obs, reward, terminated, truncated, info = eval_env.step(action)
    done = terminated or truncated

print("Actions:", actions)
print("Reward:", reward)
print("Latency:", info.get("total_latency"))

eval_env.close()

Actions: [0, 0, 0, 0]
Reward: 0.5
Latency: 419430.4


In [232]:
from copy import deepcopy
import torch
from torch.distributions import Categorical


def train_reinforce(
    env,
    policy,
    optimizer,
    num_episodes=1000,
    use_mask=True,
):
    policy.train()
    device = next(policy.parameters()).device

    reward_history = []
    running_baseline = 0.5

    best_schedule = deepcopy(env.baseline_schedule)
    best_reward = 0.5
    best_latency = env.baseline_latency

    for episode in range(num_episodes):
        obs, _ = env.reset()
        log_probs = []

        # Collect one complete episode.
        while True:
            obs_tensor = torch.as_tensor(
                obs, dtype=torch.float32, device=device
            )
            logits = policy(obs_tensor)

            if use_mask:
                mask = torch.as_tensor(
                    env.action_masks(),
                    dtype=torch.bool,
                    device=device,
                )
                logits = logits.masked_fill(~mask, float("-inf"))

            distribution = Categorical(logits=logits)
            action = distribution.sample()
            log_probs.append(distribution.log_prob(action))

            obs, reward, terminated, truncated, info = env.step(
                action.item()
            )

            if terminated or truncated:
                break

        # Intermediate rewards are zero in our environment.
        advantage = reward - running_baseline
        loss = -advantage * torch.stack(log_probs).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_baseline = (
            0.95 * running_baseline + 0.05 * reward
        )
        reward_history.append(reward)

        if info.get("valid", False) and reward > best_reward:
            best_schedule = deepcopy(env.schedule)
            best_reward = reward
            best_latency = info["total_latency"]

        if (episode + 1) % 100 == 0:
            mean_reward = sum(reward_history[-100:]) / 100
            print(
                f"Episode {episode + 1}: "
                f"average reward = {mean_reward:.4f}"
            )

    return {
        "reward_history": reward_history,
        "best_schedule": best_schedule,
        "best_reward": best_reward,
        "best_latency": best_latency,
    }


def evaluate(
    env,
    policy,
    use_mask=True,
    deterministic=True,
):
    policy.eval()
    device = next(policy.parameters()).device

    obs, _ = env.reset()
    actions = []

    with torch.no_grad():
        while True:
            obs_tensor = torch.as_tensor(
                obs, dtype=torch.float32, device=device
            )
            logits = policy(obs_tensor)

            if use_mask:
                mask = torch.as_tensor(
                    env.action_masks(),
                    dtype=torch.bool,
                    device=device,
                )
                logits = logits.masked_fill(~mask, float("-inf"))

            distribution = Categorical(logits=logits)

            if deterministic:
                action = distribution.probs.argmax()
            else:
                action = distribution.sample()

            actions.append(action.item())

            obs, reward, terminated, truncated, info = env.step(
                action.item()
            )

            if terminated or truncated:
                break

    valid = info.get("valid", False)

    return {
        "actions": actions,
        "valid": valid,
        "reward": reward,
        "latency": info.get("total_latency"),
        "schedule": deepcopy(env.schedule) if valid else None,
    }

It still facing a problem that it prefer keep

Now let's add more context into the observation. 

- fast_memory_capacity
- slow_memory_bandwidth
- native_granularity
- Operation base_costs

In [230]:
from copy import deepcopy
import gymnasium as gym
from gymnasium import spaces
from evaluator import Evaluator
import numpy as np

class SchedulingEnv(gym.Env):
    def __init__(self, problem):
        super().__init__()
        
        self.problem = deepcopy(problem)
        self.evaluator = Evaluator(self.problem)
        
        baseline = self.evaluator.baseline()
        self.baseline_schedule = baseline["schedule"]
        # We dont need the subgraph latency because the agent will change it
        # the subgraph latency here belongs to the original graph
        self.baseline_schedule.pop("subgraph_latencies", None)
        self.baseline_latency = baseline["evaluation"]["total_latency"]

        # Observation space
        self.num_ops = len(self.problem["op_types"])
        self.observation_space = spaces.Box(
            low=0.0,
            high=np.inf,
            shape=(self.num_ops+5,),
            dtype=np.float32
        )
        
        # Action space
        self.action_space = spaces.Discrete(2)
    
    def action_masks(self):
        if self.terminated:
            return np.array([False, False], dtype=bool)

        candidate = self.create_fusion_candidate()
        evaluation = self.evaluator.evaluate(candidate)

        return np.array(
            [True, evaluation["valid"]],  # [KEEP, FUSE]
            dtype=bool,
        )
    def get_observation(self):
        observation = np.zeros(
            self.num_ops + 5,
            dtype=np.float32,
        )

        scale = max(1, self.num_ops - 1)

        # Current grouping.
        for group_index, group in enumerate(self.schedule["subgraphs"]):
            for op in group:
                observation[op] = group_index / scale

        # Current decision position.
        observation[self.num_ops] = self.boundary_index / scale

        # Hardware context, using fixed scaling references.
        observation[self.num_ops + 1:] = [
            self.problem["fast_memory_capacity"] / 1_000_000,
            self.problem["slow_memory_bandwidth"] / 100,
            self.problem["native_granularity"][0] / 128,
            self.problem["native_granularity"][1] / 128,
        ]

        return observation
        
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.schedule = deepcopy(self.baseline_schedule)
        self.boundary_index = 0
        # If there is only one operation then we dont need to fuse
        self.terminated = len(self.problem["op_types"]) <= 1
        
        observation = self.get_observation()
        
        info = {"baseline_latency": self.baseline_latency}
        
        return observation, info
    
    def create_fusion_candidate(self):
        '''
        [1] [2] | [3] [4] [5]
        
        i = 1
        [1] [2,3] [4] [5]
        '''
        candidate = deepcopy(self.schedule)
        i = self.boundary_index
        
        # Combine the adjacent groups
        left = candidate["subgraphs"][i]
        right = candidate["subgraphs"][i+1]
        
        merged = left + right
        candidate["subgraphs"][i:i+2] = [merged]
        
        # A simple trial tile for our current graph.
        # Check if the merged op having Matmul
        has_matmul = any(
            self.problem["op_types"][op] == "MatMul"
            for op in merged
        )
        tile = [128, 128, 64 if has_matmul else 1]

        candidate["granularities"][i:i + 2] = [tile]
        candidate["tensors_to_retain"][i:i + 2] = [[]]
        
        return candidate

    
    def step(self, action):
        if self.terminated:
            raise RuntimeError("Episode finished. Call reset() first")
        
        if not self.action_space.contains(action):
            raise ValueError("Action must be 0 or 1.")
        
        info = {"baseline_latency": self.baseline_latency}
        
        # Keep the boundary
        if action == 0:
            self.boundary_index += 1
        elif action == 1:
            candidate = self.create_fusion_candidate()
            evaluation = self.evaluator.evaluate(candidate)
            
            # If the candidate is not valid
            if not evaluation["valid"]:
                self.terminated = True
                info.update(evaluation)
                return self.get_observation(), -1.0, True, False, info

            # The candidate is valid
            self.schedule = candidate
            
            
        self.terminated = self.boundary_index >= len(self.schedule["subgraphs"]) - 1
        reward = 0.0

        if self.terminated:
            feedback = self.evaluator.reward(self.schedule)
            reward = feedback["reward"]
            info.update(feedback["evaluation"])
        
        return (
            self.get_observation(),
            reward,
            self.terminated,
            False,  # No external time limit, so not truncated.
            info
        )
    
    def close(self):
        self.evaluator.close()

In [242]:
import torch
from torch import nn
from torch.distributions import Categorical
from pathlib import Path
import json
import sys

ROOT = graph_path.resolve().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from evaluator import Evaluator

torch.manual_seed(42)

graph_path = Path("../benchmarks/mlsys-2026-1.json")
problem = json.loads(graph_path.read_text())

evaluator = Evaluator(problem)
env = SchedulingEnv(problem)

num_inputs = env.observation_space.shape[0] # 6
num_actions = env.action_space.n # 2 keep or fuse
obs, info = env.reset()

num_inputs = env.observation_space.shape[0]
num_actions = env.action_space.n

assert num_inputs == env.num_ops + 5

policy = nn.Sequential(
    nn.Linear(num_inputs, 64),
    nn.Tanh(),
    nn.Linear(64, num_actions),
)

optimizer = torch.optim.Adam(
    policy.parameters(),
    lr=0.001,
)

print("Observation size:", num_inputs)
print(policy)

Observation size: 10
Sequential(
  (0): Linear(in_features=10, out_features=64, bias=True)
  (1): Tanh()
  (2): Linear(in_features=64, out_features=2, bias=True)
)


In [244]:
training = train_reinforce(
    env,
    policy,
    optimizer,
    num_episodes=1000,
    use_mask=False,
)

print("Best groups:", training["best_schedule"]["subgraphs"])
print("Best reward:", training["best_reward"])
print("Best latency:", training["best_latency"])


eval_env = SchedulingEnv(problem)

result = evaluate(
    eval_env,
    policy,
    use_mask=True,
    deterministic=True,
)

print("Actions:", result["actions"])
print("Valid:", result["valid"])
print("Reward:", result["reward"])
print("Latency:", result["latency"])

eval_env.close()

Episode 100: average reward = 0.5000
Episode 200: average reward = 0.5000
Episode 300: average reward = 0.5000
Episode 400: average reward = 0.5000
Episode 500: average reward = 0.5000
Episode 600: average reward = 0.5000
Episode 700: average reward = 0.5000
Episode 800: average reward = 0.5000
Episode 900: average reward = 0.5000
Episode 1000: average reward = 0.5000
Best groups: [[0], [1], [2], [3], [4]]
Best reward: 0.5
Best latency: 419430.4
Actions: [0, 0, 0, 0]
Valid: True
Reward: 0.5
Latency: 419430.4


My idea is that the current encoding of the input is too simple. We should find the better way.
 
A way that I found is that using GraphSAGE like in the paper "Transferable Graph Optimizers for ML Compilers" or we could call it GO.

Now let's set up to use it in our environment

## 4. Explore tiling

Tiling chooses the size of one execution chunk for a scheduled subgraph: `[w, h, k]`. `w × h` is the output slice. For MatMul, `k` is the reduction slice; for Pointwise, `k` is effectively 1. On benchmark 1, the first operation is a MatMul that produces a 512 × 512 tensor. A 128 × 128 output tile gives a 4 × 4 grid (16 spatial tiles); a 64 × 64 tile gives an 8 × 8 grid (64 spatial tiles). Smaller `k` means more reduction slices.

A smaller tile uses less fast memory, but may need more passes and may underuse the native 128 × 128 spatial array. A larger tile can exceed the 60,000-element fast-memory capacity. This test holds all subgraph groups and their other tiles fixed, changing only the tile for operation 0. It uses our pinned evaluator, so these are local modeled latencies rather than official leaderboard results.


In [1]:
from copy import deepcopy
from pathlib import Path
import json
import sys

project_root = Path.cwd().resolve()
if not (project_root / "benchmarks").is_dir():
    project_root = project_root.parent
if not (project_root / "benchmarks").is_dir():
    raise RuntimeError("Run from the repository root or rl_scheduler/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluator import Evaluator

problem_1 = json.loads((project_root / "benchmarks/mlsys-2026-1.json").read_text())
trial_tiles = [
    [256, 256, 256], [128, 128, 128], [128, 128, 64],
    [64, 64, 128], [64, 64, 64], [32, 32, 32],
]
with Evaluator(problem_1) as tile_evaluator:
    base_schedule = tile_evaluator.baseline()["schedule"]
    base_schedule.pop("subgraph_latencies", None)
    print("Model:", tile_evaluator.configuration["model_id"])
    print("Fast-memory capacity:", problem_1["fast_memory_capacity"], "elements")
    print("tile [w,h,k] | valid | working set | MatMul step latency | graph latency")
    for tile in trial_tiles:
        candidate = deepcopy(base_schedule)
        candidate["granularities"][0] = tile
        result = tile_evaluator.evaluate(candidate)
        if result["valid"]:
            working_set = result["working_sets"][0]
            step_latency = round(result["subgraph_latencies"][0], 1)
            graph_latency = round(result["total_latency"], 1)
            print(f"{str(tile):17} | yes   | {working_set:11} | {step_latency:18} | {graph_latency}")
        else:
            print(f"{str(tile):17} | no    | {'-':>11} | {'-':>18} | - ({result['code']})")


Model: vnrc-1f19c2f-validated-v1
Fast-memory capacity: 60000 elements
tile [w,h,k] | valid | working set | MatMul step latency | graph latency
[256, 256, 256]   | no    |           - |                  - | - (invalid_tiling)
[128, 128, 128]   | yes   |       49152 |           117964.8 | 419430.4
[128, 128, 64]    | yes   |       32768 |           117964.8 | 419430.4
[64, 64, 128]     | yes   |       20480 |           222822.4 | 524288.0
[64, 64, 64]      | yes   |       12288 |           222822.4 | 524288.0
[32, 32, 32]      | yes   |        3072 |           519321.6 | 820787.2


The working set for a single MatMul tile is approximately `h×k + k×w + h×w` tensor elements: two input slices plus the output slice. Here, `[128,128,128]` needs 49,152 elements, while `[64,64,64]` needs 12,288. Compare the valid rows above: the smaller spatial tile saves memory but increases modeled latency on this graph. `[128,128,64]` cuts the working set with no latency change under our pinned evaluator convention. The next RL step is to offer several **valid** tile choices as actions alongside fusion, then compare their results using this same evaluator.
